# Candidate Ranking Engine

## Objective

This notebook combines:

- Semantic Similarity
- Skill Matching
- Education Matching
- Experience Matching

to generate a final candidate ranking score.

The ranking engine serves as the core decision-making component of the AI Resume Screening System.

In [1]:
import pandas as pd
import numpy as np

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

c:\Users\KIIT\OneDrive\Desktop\TalentMatch_AI\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Load Processed Resume Dataset

In [2]:
df = pd.read_csv(
    "../data/processed/nlp_processed_resumes.csv"
)

df.shape

(2458, 7)

# Load Transformer Model

In [3]:
model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1416.93it/s]


# Create Job Description Text

In [5]:
import json

with open(
    "../data/processed/jd_profile.json",
    "r"
) as f:

    jd_profile = json.load(f)

In [6]:
jd_text = " ".join(

    jd_profile["Required_Skills"]

) + " " + " ".join(

    jd_profile["Education"]

) + " " + " ".join(

    jd_profile["Experience"]

)

jd_text

'Python Machine Learning Deep Learning TensorFlow AWS SQL b.tech m.tech 2+ years'

# Generate Resume Embeddings

In [7]:
sample_df = df.head(500).copy()

In [8]:
resume_embeddings = model.encode(

    sample_df["Cleaned_Resume"].tolist(),

    show_progress_bar=True

)

Batches: 100%|██████████| 16/16 [00:11<00:00,  1.38it/s]


# Generate Job Description Embedding

In [9]:
jd_embedding = model.encode(
    [jd_text]
)

# Semantic Similarity Calculation

In [10]:
semantic_scores = cosine_similarity(

    jd_embedding,

    resume_embeddings

)[0]

# Add Semantic Scores

In [11]:
sample_df["Semantic_Score"] = semantic_scores

# Dynamic Skill Matching

In [12]:
required_skills = [

    skill.lower()

    for skill in jd_profile["Required_Skills"]

]

In [13]:
def get_skill_match(resume_text):

    resume_text = str(resume_text).lower()

    matched = []

    for skill in required_skills:

        if skill.lower() in resume_text:

            matched.append(skill)

    return matched

In [14]:
sample_df["Matched_Skills"] = (

    sample_df["Cleaned_Resume"]

    .apply(get_skill_match)

)

# Skill Match Percentage

In [15]:
sample_df["Skill_Match"] = (

    sample_df["Matched_Skills"]

    .apply(

        lambda x:

        len(x)

        /

        len(required_skills)

        * 100

    )

)

# Education Matching

In [16]:
education_required = [

    edu.lower()

    for edu in jd_profile["Education"]

]

In [17]:
def education_match(text):

    text = str(text).lower()

    for edu in education_required:

        if edu in text:

            return 100

    return 0

In [18]:
sample_df["Education_Match"] = (

    sample_df["Cleaned_Resume"]

    .apply(education_match)

)

# Experience Matching

In [19]:
sample_df["Experience_Match"] = 50

# Final Candidate Score

In [20]:
sample_df["Final_Score"] = (

    0.60 * sample_df["Semantic_Score"] * 100

    +

    0.25 * sample_df["Skill_Match"]

    +

    0.10 * sample_df["Experience_Match"]

    +

    0.05 * sample_df["Education_Match"]

)

# Candidate Ranking

In [21]:
ranked_candidates = (

    sample_df

    .sort_values(

        by="Final_Score",

        ascending=False

    )

)


# Top 10 Candidates

In [22]:
ranked_candidates[

    [

        "ID",

        "Category",

        "Semantic_Score",

        "Skill_Match",

        "Final_Score",

        "Matched_Skills"

    ]

].head(10)

,ID,Category,Semantic_Score,Skill_Match,Final_Score,Matched_Skills
261,29051656,INFORMATION-TECHNOLOGY,0.323972,33.333333,32.771682,"[aws, sql]"
296,20824105,INFORMATION-TECHNOLOGY,0.320266,33.333333,32.549283,"[python, aws]"
127,12415691,DESIGNER,0.234243,33.333333,27.387944,"[python, sql]"
307,16186411,INFORMATION-TECHNOLOGY,0.276036,16.666667,25.728812,[sql]
309,26746496,INFORMATION-TECHNOLOGY,0.273252,16.666667,25.561766,[sql]
332,79541391,INFORMATION-TECHNOLOGY,0.269215,16.666667,25.319590,[sql]
295,83816738,INFORMATION-TECHNOLOGY,0.197934,33.333333,25.209387,"[aws, sql]"
314,30223363,INFORMATION-TECHNOLOGY,0.263634,16.666667,24.984733,[sql]
297,33381211,INFORMATION-TECHNOLOGY,0.180180,33.333333,24.144129,"[aws, sql]"
241,13385306,INFORMATION-TECHNOLOGY,0.249315,16.666667,24.125565,[sql]


In [23]:
ranked_candidates.to_csv(
    "../data/processed/ranked_candidates.csv",
    index=False
)

print("Ranked candidates saved successfully.")

Ranked candidates saved successfully.
